In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 量子ビットの初期化

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit-ibm-runtime~=0.43.1
```
</details>
IBM&reg; 量子処理ユニット（QPU）で回路が実行される際、量子ビットがゼロに初期化されることを保証するために、通常は回路の先頭に暗黙的なリセットが挿入されます。これは `init_qubits` フラグによって制御され、[プリミティブ実行オプション](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2)として設定されます。

ただし、リセット処理は完全ではないため、状態準備エラーが発生します。このエラーを軽減するために、システムは回路間に繰り返し遅延時間（`rep_delay`）も挿入します。各バックエンドにはデフォルトの `rep_delay` があり、通常は環境が量子ビットをリセットできるよう T1 より長く設定されています。デフォルトの `rep_delay` は `backend.default_rep_delay` を実行することで確認できます。

すべての IBM QPU は動的繰り返しレート実行を使用しており、ジョブごとに `rep_delay` を変更することができます。プリミティブジョブで送信した回路は、QPU での実行のためにまとめてバッチ処理されます。これらの回路は、要求されたショットごとに回路を繰り返し処理することで実行されます。実行は、以下の図に示すように、回路とショットの行列に対して列方向に行われます。

![最初の列はショット 0 を表します。回路は 0 から 3 の順に実行されます。2 番目の列はショット 1 を表します。回路は 0 から 3 の順に実行されます。残りの列も同じパターンに従います。](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "列方向の実行行列")

`rep_delay` は回路間に挿入されるため、実行の各ショットでこの遅延が発生します。したがって、`rep_delay` を低くすると QPU の合計実行時間は短くなりますが、以下の画像に示すように、状態準備エラー率が増加するというトレードオフがあります。

![この画像は、`rep_delay` の値を下げると状態準備エラー率が増加することを示しています。](../docs/images/guides/repetition-rate-execution/rep_delay.avif "繰り返し遅延とエラー率の関係")

`rep_delay=0` と `init_qubits=False` の両方を設定すると、量子ビットが前のショットの最終状態から開始するため、回路が実質的に「結合」されます。

なお、プリミティブジョブの回路は QPU 実行のためにバッチ処理されますが、PUB からの回路が実行される順序は保証されません。したがって、`pubs=[pub1, pub2]` を送信した場合でも、`pub1` の回路が `pub2` の回路より先に実行されるとは限りません。また、同じジョブの回路が QPU 上で単一のバッチとして実行されることも保証されません。

## プリミティブジョブの rep_delay を指定する

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## 次のステップ
> **Tip:** - [量子近似最適化アルゴリズム（QAOA）](/tutorials/quantum-approximate-optimization-algorithm)チュートリアルの例を試してみてください。
>   - [プリミティブの使い方](./get-started-with-primitives)を確認してください。